<a href="https://colab.research.google.com/github/solankiboy939/Deep_Learning/blob/main/GAN_on_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Simple GAN on MNIST (TensorFlow/Keras)
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 1️⃣ Load and preprocess MNIST
# -----------------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = (x_train.astype("float32") - 127.5) / 127.5  # Normalize to [-1, 1]
x_train = x_train.reshape(x_train.shape[0], 784)

In [ ]:
# Generator models/
# -----------------------------
def build_generator():
    model = tf.keras.Sequential([
        layers.Dense(128, activation="relu", input_shape=(100,)),
        layers.Dense(784, activation="tanh")
    ])
    return model

# -----------------------------
# 3️⃣ Discriminator model
# -----------------------------
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Dense(128, activation="relu", input_shape=(784,)),
        layers.Dense(1, activation="sigmoid")
    ])
    return model

# -----------------------------

In [ ]:
# Create models
# -----------------------------
generator = build_generator()
discriminator = build_discriminator()

discriminator.compile(loss="binary_crossentropy",
                      optimizer="adam", metrics=["accuracy"])

# Combined GAN (Generator + Discriminator)
discriminator.trainable = False
gan_input = tf.keras.Input(shape=(100,))
generated_img = generator(gan_input)
gan_output = discriminator(generated_img)
gan = tf.keras.Model(gan_input, gan_output)
gan.compile(loss="binary_crossentropy", optimizer="adam")

# -----------------------------

In [ ]:
#  Training Loop
# -----------------------------
def train_gan(epochs=3000, batch_size=128):
    half_batch = batch_size // 2

    for epoch in range(epochs):
        # ---- Train discriminator ----
        idx = np.random.randint(0, x_train.shape[0], half_batch)
        real_imgs = x_train[idx]

        noise = np.random.normal(0, 1, (half_batch, 100))
        fake_imgs = generator.predict(noise)

        d_loss_real = discriminator.train_on_batch(real_imgs, np.ones((half_batch, 1)))
        d_loss_fake = discriminator.train_on_batch(fake_imgs, np.zeros((half_batch, 1)))
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        # ---- Train generator ----
        noise = np.random.normal(0, 1, (batch_size, 100))
        valid_y = np.ones((batch_size, 1))
        g_loss = gan.train_on_batch(noise, valid_y)

        # ---- Print progress ----
        if epoch % 200 == 0:
            print(f"Epoch {epoch} | D loss: {d_loss[0]:.4f}, acc: {100*d_loss[1]:.2f}% | G loss: {g_loss:.4f}")
            show_generated_images(generator)

In [ ]:
#  Function to display results
# -----------------------------
def show_generated_images(generator, n=16):
    noise = np.random.normal(0, 1, (n, 100))
    generated_imgs = generator.predict(noise)
    generated_imgs = 0.5 * generated_imgs + 0.5  # Rescale to [0, 1]

    plt.figure(figsize=(4, 4))
    for i in range(n):
        plt.subplot(4, 4, i + 1)
        plt.imshow(generated_imgs[i].reshape(28, 28), cmap="gray")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# -----------------------------
# Run training
# -----------------------------
train_gan(epochs=2000, batch_size=128)

In [ ]:
from re import X
# Simple GAN on CIFAR-10 (Color Images)
# ======================================
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Load and preprocess CIFAR-10
# -----------------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train = (x_train.astype("float32") - 127.5) / 127.5  # Normalize to [-1,1]
x_train = x_train.reshape(x_train.shape[0], 32, 32, 3)

latent_dim = 100  # Random noise vector

In [ ]:
#  Build Generator
# -----------------------------
def build_generator():
    model = tf.keras.Sequential([
        layers.Dense(8*8*256, use_bias=False, input_shape=(latent_dim,)),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Reshape((8, 8, 256)),
        layers.Conv2DTranspose(128, (5,5), strides=(2,2), padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Conv2DTranspose(64, (5,5), strides=(2,2), padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Conv2DTranspose(3, (5,5), strides=(1,1), padding="same", activation="tanh")
    ])
    return model

In [ ]:
# Build Discriminator
# -----------------------------
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Conv2D(64, (5,5), strides=(2,2), padding="same", input_shape=[32,32,3]),
        layers.LeakyReLU(alpha=0.2),
        layers.Dropout(0.3),
        layers.Conv2D(128, (5,5), strides=(2,2), padding="same"),
        layers.LeakyReLU(alpha=0.2),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(1, activation="sigmoid")
    ])
    return model

In [ ]:
# Compile Models
# -----------------------------
generator = build_generator()
discriminator = build_discriminator()
discriminator.compile(loss="binary_crossentropy", optimizer=tf.keras.optimizers.Adam(0.0002, 0.5), metrics=["accuracy"])

# Combined model (for generator training)
discriminator.trainable = False
z = layers.Input(shape=(latent_dim,))
img = generator(z)
validity = discriminator(img)
gan = tf.keras.Model(z, validity)
gan.compile(loss="binary_crossentropy", optimizer=tf.keras.optimizers.Adam(0.0002, 0.5))

In [ ]:
# Helper to Show Generated Images
# -----------------------------
def show_generated_images(generator, epoch, n=9):
    noise = np.random.normal(0, 1, (n, latent_dim))
    gen_imgs = generator.predict(noise)
    gen_imgs = 0.5 * gen_imgs + 0.5  # Rescale [−1,1] → [0,1]

    plt.figure(figsize=(6,6))
    for i in range(n):
        plt.subplot(3,3,i+1)
        plt.imshow(gen_imgs[i])
        plt.axis("off")
    plt.suptitle(f"Epoch {epoch}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Training Loop
# -----------------------------
def train_gan(epochs=5000, batch_size=128):
    half_batch = batch_size // 2

    for epoch in range(epochs):
        # --- Train Discriminator ---
        idx = np.random.randint(0, x_train.shape[0], half_batch)
        real_imgs = x_train[idx]

        noise = np.random.normal(0, 1, (half_batch, latent_dim))
        fake_imgs = generator.predict(noise)

        d_loss_real = discriminator.train_on_batch(real_imgs, np.ones((half_batch, 1)))
        d_loss_fake = discriminator.train_on_batch(fake_imgs, np.zeros((half_batch, 1)))
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        # --- Train Generator ---
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        valid = np.ones((batch_size, 1))
        g_loss = gan.train_on_batch(noise, valid)

        # --- Print progress ---
        if epoch % 100 == 0:
            print(f"Epoch {epoch} | D loss: {d_loss[0]:.4f}, acc: {100*d_loss[1]:.2f}% | G loss: {g_loss:.4f}")
        if epoch % 500 == 0:
            show_generated_images(generator, epoch)

# -----------------------------
# Train the GAN
# -----------------------------
train_gan(epochs=5000, batch_size=128)